# 18 — Async vLLM CrafText rollout

Перед запуском на GPU-сервере подготовьте окружение и локальный snapshot:

```bash
uv sync --extra envs --extra prompts --extra vllm --extra examples
# artifacts/models/qwen25-05b-instruct должен существовать заранее
```

Notebook ничего не скачивает неявно. Если vLLM или snapshot отсутствуют, preflight-ячейка остановится с понятной ошибкой.

Этот notebook использует тот же `GenerationBatch/GenerationResult`, но вызывает vLLM через async collector. Каждая строка environment batch становится отдельным async request; vLLM server/runtime может собрать их во внутренний continuous batch, а среда шагает батчем после получения ответов.

In [ ]:
from __future__ import annotations

from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
from transformers import AutoTokenizer

from tunix_craftext.artifacts.replay import ReplayArtifact, ReplayStep, save_replay
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.prompts import MegaPromptRenderer, PromptContext, RenderedPrompt, compose_craftext_goal
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler
from tunix_craftext.env.text_policy import DecodedAction, decode_action_outcome
from tunix_craftext.inference import EngineProfile, GenerationBatch, VllmInferenceEngine, collect_generation_results
from tunix_craftext.models.llm import LlmRequest


In [ ]:
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')
SNAPSHOT = ROOT / 'artifacts/models/qwen25-05b-instruct'
if not SNAPSHOT.is_dir():
    raise FileNotFoundError(f'Local model snapshot is missing: {SNAPSHOT}')

CONFIG_PATH = ROOT / 'configs/mvp/qwen_craftext.yaml'
OUTPUT_DIR = ROOT / 'artifacts/trajectories/vllm-async-rollout'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
    goal_prefix=config.run.goal,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
base_renderer = MegaPromptRenderer(config.prompt.template)
tokenizer = AutoTokenizer.from_pretrained(str(SNAPSHOT), trust_remote_code=True)
print('instruction_index:', prepared_instruction_index)
print('goal:', prepared_goal)


In [ ]:
def item_at(tree, index: int):
    return jax.tree.map(lambda leaf: leaf[index], tree)


def render_qwen_prompt(state, action_mask):
    context = runtime.adapter.episode_context(state)
    goal = prepared_goal
    if context is not None:
        goal = compose_craftext_goal(
            prepared_goal,
            scenario_instruction=context.instruction,
            world_preset=context.world_preset,
            text_constraint=context.text_constraint,
        )
    rendered = base_renderer.render(
        PromptContext(
            goal,
            runtime.adapter.prompt_state(state),
            runtime.actions,
            (),
            safety='' if context is None else context.text_constraint,
            world_preset='' if context is None else context.world_preset,
        )
    )
    chat_text = tokenizer.apply_chat_template(
        [
            {
                'role': 'system',
                'content': 'You are a CrafText agent. Return exactly one <action>LABEL</action>.',
            },
            {'role': 'user', 'content': rendered.text},
        ],
        add_generation_prompt=True,
        tokenize=False,
    )
    if not isinstance(chat_text, str) or not chat_text.strip():
        raise ValueError('Qwen chat template did not return text')
    return RenderedPrompt(chat_text, runtime.actions, rendered.template_name), np.asarray(action_mask)


def decode_or_fallback(prompt, raw_text: str, action_mask, fallback_label='NOOP'):
    decision, metrics = decode_action_outcome(prompt, raw_text)
    if decision is not None and not bool(action_mask[decision.action_id]):
        decision = None
    if decision is None:
        action_id = runtime.actions.index_of(fallback_label)
        return DecodedAction(action_id, fallback_label, raw_text), metrics, True
    return decision, metrics, False


In [ ]:
engine_profile = EngineProfile(
    name='qwen25-05b-vllm-async-rollout',
    backend='vllm-offload',
    model=str(SNAPSHOT),
    tensor_parallel_size=1,
    max_model_len=2048,
    dtype='bfloat16',
    mode='async',
    policy_version=0,
    metadata={'notebook': '18_async_vllm_craftext_rollout.ipynb'},
)
engine = VllmInferenceEngine.from_profile(engine_profile)
print('engine:', engine.profile)


In [ ]:
BATCH_SIZE = 2
HORIZON = 4
MAX_NEW_TOKENS = 8
MAX_IN_FLIGHT = BATCH_SIZE

keys = jax.random.split(jax.random.PRNGKey(config.run.seed), BATCH_SIZE)
reset = jax.vmap(runtime.adapter.reset)(keys)
states = reset.state
action_masks = reset.action_mask
steps_by_env = [[] for _ in range(BATCH_SIZE)]


In [ ]:
for t in range(HORIZON):
    prompt_rows = [render_qwen_prompt(item_at(states, i), action_masks[i]) for i in range(BATCH_SIZE)]
    batches = tuple(
        GenerationBatch(
            (LlmRequest(prompt, max_new_tokens=MAX_NEW_TOKENS),),
            seed=config.run.seed + t * BATCH_SIZE + i,
            group_id=f't{t}-env{i}',
            policy_version=engine.profile.policy_version,
        )
        for i, (prompt, _mask) in enumerate(prompt_rows)
    )
    records = await collect_generation_results(engine, batches, max_in_flight=MAX_IN_FLIGHT)
    decoded = [
        decode_or_fallback(
            prompt_rows[record.index][0],
            record.result.responses[0].raw_text,
            prompt_rows[record.index][1],
        )
        for record in records
    ]
    action_ids = jnp.asarray([item[0].action_id for item in decoded], dtype=jnp.int32)
    step_keys = jax.random.split(jax.random.PRNGKey(config.run.seed + 10_000 + t), BATCH_SIZE)
    transition = jax.vmap(runtime.adapter.step)(step_keys, states, action_ids)
    for env_index, (record, (decision, metrics, fallback)) in enumerate(zip(records, decoded, strict=True)):
        response = record.result.responses[0]
        steps_by_env[env_index].append(
            ReplayStep(
                index=t,
                prompt=prompt_rows[env_index][0].text,
                raw_completion=response.raw_text,
                action_id=decision.action_id,
                action_label=decision.label,
                reward=float(transition.reward[env_index]),
                terminated=bool(transition.terminated[env_index]),
                truncated=bool(transition.truncated[env_index]),
                observation=None,
                token_ids=response.token_ids,
                token_logprobs=response.token_logprobs,
                prompt_token_ids=response.prompt_token_ids,
                fallback_used=fallback,
                masked_action=bool(metrics.masked_action),
            )
        )
    print('step', t, 'actions', [item[0].label for item in decoded], 'rewards', transition.reward.tolist())
    states = transition.state
    action_masks = transition.action_mask


In [ ]:
for env_index, steps in enumerate(steps_by_env):
    replay = ReplayArtifact(
        config_path=str(CONFIG_PATH.relative_to(ROOT)),
        commit='notebook-async-vllm',
        backend=f'{engine.profile.backend}:{engine.profile.model}',
        steps=tuple(steps),
    )
    path = OUTPUT_DIR / f'env-{env_index}.json'
    save_replay(path, replay)
    print('saved', path, 'steps=', len(steps))
